# RepairScope-X — Notebook 23 (cohort-integrity hotfix v2)
## BD3LM: fixed 32 denoising steps per block, followed by native suffix self-repair

**Why this notebook:** the BD3LM 64-token study produced 0/24 natural task successes and 0/64 successful repairs. In Notebook 20, MDLM's 128/256-length budget experiment used a fixed overall 64 NFEs; longer responses consequently received fewer denoising steps per block. This independent BD3LM experiment fixes **32 denoising steps per 32-token block**, then tests whether real BD3LM failures become repairable.

- 24 frozen problems × 128 and 256 output tokens = **48 new model-native generations**.
- Up to 8 deterministically selected genuine 128-token failures × 4 suffix repair scopes × 2 seeds = **64 repairs**.
- **Notebook 23 does not depend on Notebook 21 being complete, nor on Notebook 20 BD3LM results.** It embeds its own 64-token BD3LM baseline solely for descriptive context.
- All inference uses the pinned official `BD3LMSampler.sample` and official model class. **No arbitrary masked-token infilling**: BD3LM's official sampler has no implemented `infill` method. The intervention holds a token prefix in observed context and regenerates the remaining suffix.
- Each generation or repair batch has **at most 4 trials**, with a soft 8-minute time limit **between** trials; a single slow trial cannot be interrupted by the soft limit.
- Atomic manifests + per-trial append-and-fsync under Google Drive. Repeat any batch after a disconnect to skip completed trial IDs.
- Natural-generation checkpoint ZIP is exported **before** starting self-repair.
- The 24 original problems are the previously frozen exploratory cohort; they are not an external unseen test set.

**Interpretation guardrails:** output length and total compute both vary in this study. Decoder behavior, output truncation, and weak baseline reasoning can also explain negative results. The 64-token baseline used a different configured total budget; it is not a matched-compute causal comparison. The 'oracle' is an in-sample upper bound. Actual GPU model execution must be verified in Colab.


### 0. Safe Python library setup — first code cell

In [ ]:
# Must be first code cell; does not import NumPy into the notebook kernel.
import os, sys, subprocess, pathlib
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN']='1'  # public model snapshots
if sys.version_info < (3,10):
    raise RuntimeError('Python 3.10 or newer is required.')
check="import numpy, torch; assert numpy.asarray([1,2]).sum()==3; print('NumPy',numpy.__version__,'Torch',torch.__version__)"
r=subprocess.run([sys.executable,'-c',check],capture_output=True,text=True)
if r.returncode:
    print(r.stderr[-7000:])
    raise RuntimeError('Scientific Python environment already corrupted. Colab: Runtime > Disconnect and delete runtime; reconnect; do NOT reinstall NumPy inside a live runtime.')
print(r.stdout)
# Narrow, pinned Python interfaces only; never reinstall PyTorch, NumPy, SciPy or Pandas.
packages=['transformers==4.57.0','accelerate==1.11.0',
          'huggingface-hub==0.35.3','tokenizers==0.22.1','safetensors==0.6.2']
subprocess.run([sys.executable,'-m','pip','install','-q','--disable-pip-version-check','--no-deps',*packages],check=True)
preflight="import torch,transformers,accelerate,huggingface_hub,tokenizers,safetensors; from transformers import AutoTokenizer; from transformers.processing_utils import Unpack; from transformers.utils import TransformersKwargs; from transformers.modeling_attn_mask_utils import _prepare_4d_attention_mask; print('model interface',transformers.__version__)"
r=subprocess.run([sys.executable,'-c',preflight],capture_output=True,text=True)
if r.returncode:
    print(r.stderr[-9000:])
    raise RuntimeError('Model library import preflight failed. Start from a fresh runtime and report this error rather than running GPU stages.')
print(r.stdout)

### 1. Colab CUDA runtime, pinned official source, persistent Google Drive checkpointing


In [ ]:
import base64, zlib, json, re, math, ast as pyast, gc, time, random, traceback, hashlib, zipfile, csv, shutil, subprocess, sys, importlib.util, os
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import torch, transformers
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
assert torch.cuda.is_available(), 'Choose T4 GPU and restart from the beginning.'
assert transformers.__version__=='4.57.0', 'Transformers version mismatch; restart with a fresh runtime.'
GPU=torch.cuda.get_device_name(0); VRAM=torch.cuda.get_device_properties(0).total_memory/2**30
assert VRAM >= 7, f'Insufficient GPU memory: {VRAM:.2f} GB'
print('GPU',GPU,'VRAM',round(VRAM,2),'GB','torch',torch.__version__)
from google.colab import drive
drive.mount('/content/drive')
DRIVE=Path('/content/drive/MyDrive/RepairScopeX/RSX23_BD3LMCostControlled_v1')
DRIVE.mkdir(parents=True,exist_ok=True)
# Verify persistent destination is writable; do not fall back to ephemeral /content.
(DRIVE/'drive_write_test.txt').write_text('ok',encoding='utf-8')
assert (DRIVE/'drive_write_test.txt').read_text()=='ok'
SOURCE_COMMIT='ca176752fbceec49c6b4777a2c18ae88e4eb10ed'
ROOT=Path('/content/rsx_v6_official_repo')
if not (ROOT/'.git').exists():
    ROOT.mkdir(parents=True,exist_ok=True)
    subprocess.run(['git','init','-q',str(ROOT)],check=True)
    subprocess.run(['git','-C',str(ROOT),'remote','add','origin','https://github.com/ZHZisZZ/dllm.git'],check=True)
try:
    sha=subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],stderr=subprocess.DEVNULL,text=True).strip()
except subprocess.CalledProcessError:
    sha=''
if sha!=SOURCE_COMMIT:
    for retry in range(3):
        try:
            subprocess.run(['git','-C',str(ROOT),'fetch','--quiet','--depth','1','origin',SOURCE_COMMIT],check=True,timeout=240)
            subprocess.run(['git','-C',str(ROOT),'checkout','-q','--force','FETCH_HEAD'],check=True,timeout=80)
            break
        except Exception:
            if retry==2: raise
            time.sleep(2**retry)
sha=subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()
assert sha==SOURCE_COMMIT
print('Official code SHA',sha)


### 2. Load byte-identical official sampler source with inference-only package initializers

In [ ]:
# Exact official sampler files, copied BYTE-FOR-BYTE from the pinned upstream commit.
SHIM=Path('/content/rsx_v6_inference_only')
files=[
    'dllm/core/samplers/__init__.py','dllm/core/samplers/base.py',
    'dllm/core/samplers/mdlm.py','dllm/core/samplers/bd3lm.py',
    'dllm/core/samplers/utils.py','dllm/core/schedulers/__init__.py',
    'dllm/core/schedulers/alpha.py','dllm/core/schedulers/kappa.py',
]
for rel in files:
    source=ROOT/rel; destination=SHIM/rel
    assert source.exists(),source
    destination.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(source,destination)
    assert hashlib.sha256(source.read_bytes()).digest()==hashlib.sha256(destination.read_bytes()).digest()
for rel in ['dllm/__init__.py','dllm/core/__init__.py']:
    f=SHIM/rel;f.parent.mkdir(parents=True,exist_ok=True)
    f.write_text('# Inference namespace, without unrelated training imports.\n')
# Model class from exact official source, no checkpoint `trust_remote_code` needed.
MODEL_FILE=ROOT/'dllm/pipelines/a2d/models/qwen2/modeling_qwen2.py'
assert MODEL_FILE.is_file()
sys.path.insert(0,str(SHIM))
from dllm.core.samplers import MDLMSampler, MDLMSamplerConfig, BD3LMSampler, BD3LMSamplerConfig
model_module_name='rsx_v6_a2d_qwen2_source'
if model_module_name not in sys.modules:
    spec=importlib.util.spec_from_file_location(model_module_name,str(MODEL_FILE))
    module=importlib.util.module_from_spec(spec)
    sys.modules[model_module_name]=module
    spec.loader.exec_module(module)
else:
    module=sys.modules[model_module_name]
A2DQwen2Config=module.A2DQwen2Config
A2DQwen2LMHeadModel=module.A2DQwen2LMHeadModel
assert A2DQwen2Config.model_type=='a2d-qwen2'
print('Official samplers:',MDLMSampler.__name__,BD3LMSampler.__name__)

### 3. Frozen 24-problem benchmark (identical to NB20)

In [ ]:
PAYLOAD_SHA="cc04b82051f62f8640b92c418571d5efd3fd89334bf4be126bfae6f508ff5328"
_encoded="eNq1WW2P27gR/iuEsUW+eBeWLPkNRYttkyAX3KXFbooULQofLdMWYUlUScqOcrj/3mdIyZad7a1tnIFssmtyhjPPPPO2+aVnVKUTYXqzf//SK7VaZCKfy2Vv1ktUVdil2hWzhGdyobmVqpgFk0E8iwa9fm/JLTfCdm/OV1qYFGcl11bSfTo9SNOJVnlJQv8wgtlUsKLKF0IbFk77DH94sWQTJr7yxGY1U0WCW4pBalklXsByvRaWBeEDexK20gXjbIs3loxradNcWJlAQQlTDN50GgW+djhkP7//4dPjj/PHT89f3j3N2B/963/6+QGmecW9GTQPfu2fj8bk9wcjiPosGHg0wuFZcNwMjcvAiMY3oEafDT0W8TlQDAe3wgKaLwIjvAE1kCPBuGHG4Bw44vBmeJDqiwAJbsCOYNTJlbNKxzC6GUGiy/CIb5IsYXQJGvHN0IgvRGMY/P5wDMGMsGFHcFa+RDdLl+jS8jG9AT2IH/El9eM+HN4KEFL9PSJrk082M6u5LGbxcHQMgjs8z/WfuLUs5QZWl4IXlWVlxgvr7sFBbpk0LB4M2ErAzUW9//6BsYCZ/1ZcC7ZSyjK1ajQYlvCC5XwjcJmtNc9N5xByP6yYtEBuIwzBe3qDIPbSp0dsUVkrtOcp26y/PzEiywzs0ewORE7VjuVVkrKlwlOIllO7gu8shVsHT//cLU9BGA9eQXw0GsfXIv6R5zBmUdWGjZi9N6nU1pscD/4Aj1ZA6HMqaiY4LE+UseyOuhP7sPcG5IIzJa+7do9eM3ocDMdXG63Sgi1cxIIQRiWKQM95wrVShel7Y3dCrlNZrBE5pCEG5gf2waVJAWOTDUhWMAHKZE2mUfysC0ujh8mCRfBvtRJaFIiqRibj3TVe0IIva4fTUmRyK3T9wL6QaihYaClWyNBM8C1MJJ0baRM6pXyVycZ9VqaqEH0X+0xam5GgwoFmz1ZsBZywXhhvS/AaXlp86xgIydbcg1WIypcmR1wxUBa+ORRsn3zxKPRbQS1yBIIA+g7BbiSj+LWEH4ejayP5meygT1ujeE4lk37K4WPN/so177OPvBDWV8CPQusaNQIAwc9oFs9GD5TDB49PVViibytwN0YlhTFJlXHri2amjHHveP30FOphlvn8lCuvIFH5QhZOQupWs6LMYUrzYi2ME6eUpzs5QyAmLoV8lKqakC61TES3nk5fze3o6tx+hkFauXpq5FdmJaU6fsh5UbMlr8AL6i90jJg7FGFpUmlie1Y7QZtqIdx53xcwkk1SmS1xaV/JCN0uZ8LgVc6Mr87+90g6gnTFcwkrtagK3OizvwuqudK0WAcjVoIJS5cwCwTsHjkA05ON8I0v5dnKd5WGMjhPeb6o4ISmbPog3hi2VqSsCXWIyCMVD3pT9JqlWhNQ/CVpnhl1rMIgtzVoauTSFQfesM0lamtTNxHaJ9qiSxE4GIBut/QKPMuOmser7AriyfVF2Igybbq1Fist4Te3SiNP2Y5TLFLh/iFYEZZUVUb4JKbKm4nEIhuY2oqmt6MImYo4Wqod9VXbqb3QTAX8829kOe43NqEX+R7mq6fQ65rh7SVNDL7yHaz11ZJI7vOjg73vagdN3qyuoq6b+yp07Bmutw7tlMaUdUczi6TBDIbbtM/sDjWB7f6f/V729DWn97SSHSHzqvUYM9d8CaKYo18ewLzXkjeKwquTF7UykeZNm71En5BVmyoDbHAoYutKwhbTEIUlQls82unRW6kySfXqHXX5RtQpipgB8MW6PfOq3NHo5MgrOZU66iOHF5tzQpbyGDcOBh+Z25jmOtOgFesUzvbc1c3voTgpoi+EwWx5XjZhSFAr7qej4VEo3IXzQvG4Ih6BTVScOPFK25rlgme+4+0o71C0jPDs+kuVbHAVNQ7Eyjj6GKWjRko/AVGR0byjC9DLpWIUw1sjkWE0geduVXlRC3OjPfipBeCpAA6vXSQm4RTztCoRXKfJaX5JAS0vslHCMaBOYtJhDtVyb8hv62l3jOXxKDueROdEIjj5TckFkXCMLJVffUCcVUa56uZPaRuUGa020eDo0PH0odsVMB3SPeskqMhkWdeVYHAWpybBtZ5gzdHAt6QBvJ2IFILvuuYaVbY4nHkL3XDEzYZ2OJ9+ja+8SUPkXxDvpY4kDp6/orqLQXwOBOPB1RC8o2WAOMSeavQ5Uwrq1CNqf4gaOObShHJOFOtMmtQv9i+cY1YpUKU7AXYw+ovt6OX0v6jWTyTC33xB79GQfw4ow8HVDD9QGAzpbFRKbTxLRp1Pc7WVh1C/YYnm3zA3YcbB30mqVIYPjcCW5Ut2rSq3j7Eg2u9wXgVpRtcEKXA43c/i7tVjUP1nDrJG1m0MreqLCRRPo2uxeltp2bSZR73hheHekc/iKw3kghqbK6i48MS/Kb3gtFH63Z2ZVJXwaynYHQuROIva7SR+D24X/IejRdFt9cDGbc3NlSNyDM+qGtH0Wo9/rDY0i0hqHkFEY7ukPRcEcCZpGnbJQO4cbxeVMuM1CcRjf8P4K2i6W+lvdiuj09j8rsIk6lAh6OJR2w2n4VmtNxpd6++/4BWnIgFzhqhtlUnvq9JHeQi6Y09vat26zlmScUO/hnJ98S3fQogE40mzi7TSC2zwkzHSHLf3Ohyjv/n3TgvJXtJd2ouQ8r2F6mifIFz+0+8ZN+LCqTlmA2AznATTfvPfknOuUV+2Am4+iZJL/ZyoUvxz/klZQUkWBPO3yKh8/rddMX/PZVZpYeb4qjLw8pskCHNhU0WYr+RXRPj5w+N9GI+wAxSbPpugzGvW4v7r/wDPufiP"
_raw=zlib.decompress(base64.b64decode(_encoded))
assert hashlib.sha256(_raw).hexdigest()==PAYLOAD_SHA
SOURCES=json.loads(_raw.decode('utf-8'))['sources']
assert len(SOURCES)==24 and len(set(x['problem_id'] for x in SOURCES))==24
assert Counter(x['dataset'] for x in SOURCES)=={'countdown_fresh':8,'gsm8k':8,'svamp':8}
print('Frozen sources:',Counter(x['dataset'] for x in SOURCES))

### 4. Task-aware answer verifier and regression tests

In [ ]:
# Shared benchmark evaluator. Countdown requires a legal expression with all numbers once.
FINAL_RE=re.compile(r'(?im)^\s*FINAL_ANSWER\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$')
NUM_RE=re.compile(r'[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?')
PROMPT_RE=re.compile(r'Use the numbers\s+(.+?)\s+exactly once\s+to produce the target\s+([-+]?(?:\d+(?:\.\d*)?|\.\d+))',re.I)
ARITH_RE=re.compile(r'[0-9eE+\-*/().=\s]{3,}')

def equals(a,b):
    return a is not None and b is not None and math.isclose(float(a),float(b),rel_tol=1e-8,abs_tol=1e-8)

def lastnum(text):
    x=NUM_RE.findall(str(text).replace(',',''))
    try:return float(x[-1]) if x else None
    except:return None

def eval_arith(expr):
    try:tree=pyast.parse(expr,mode='eval')
    except (SyntaxError,ValueError):return None
    numbers=[]
    def visit(node):
        if isinstance(node,pyast.Expression):return visit(node.body)
        if isinstance(node,pyast.Constant) and type(node.value) in (int,float):
            v=float(node.value)
            if not math.isfinite(v):raise ValueError('nonfinite')
            numbers.append(v);return v
        if isinstance(node,pyast.BinOp) and type(node.op) in (pyast.Add,pyast.Sub,pyast.Mult,pyast.Div):
            a,b=visit(node.left),visit(node.right)
            if isinstance(node.op,pyast.Add):return a+b
            if isinstance(node.op,pyast.Sub):return a-b
            if isinstance(node.op,pyast.Mult):return a*b
            if b==0:raise ZeroDivisionError()
            return a/b
        if isinstance(node,pyast.UnaryOp) and type(node.op) in (pyast.UAdd,pyast.USub):
            a=visit(node.operand);return -a if isinstance(node.op,pyast.USub) else a
        raise ValueError('forbidden AST')
    try:
        ans=visit(tree)
        return (ans,numbers) if math.isfinite(ans) else None
    except (ZeroDivisionError,ValueError,OverflowError):return None

def countdown_check(text,prompt):
    m=PROMPT_RE.search(prompt)
    if not m: raise ValueError('Unexpected Countdown prompt: '+prompt[:200])
    required=[float(x) for x in NUM_RE.findall(m.group(1))];goal=float(m.group(2))
    normalized=str(text).replace('×','*').replace('÷','/').replace('\\times','*')
    for chunk in ARITH_RE.findall(normalized):
        if not any(k in chunk for k in '+-*/'):continue
        for candidate in chunk.split('='):
            candidate=candidate.strip();value=eval_arith(candidate)
            if value is None:continue
            output,nums=value
            if len(nums)==len(required) and all(equals(a,b) for a,b in zip(sorted(nums),sorted(required))) and equals(output,goal):
                return True,candidate
    return False,None

def verify(text,s):
    strict_matches=FINAL_RE.findall(str(text))
    strict=float(strict_matches[-1]) if strict_matches else None
    numeric=lastnum(text)
    if s['dataset']=='countdown_fresh': success,expr=countdown_check(text,s['prompt'])
    else: success=equals(numeric,s['target']);expr=None
    return {'task_success':bool(success),'strict_success':equals(strict,s['target']),
            'numeric_success':equals(numeric,s['target']),'strict_prediction':strict,
            'numeric_prediction':numeric,'valid_expression':expr}

_demo='Use the numbers 19, 29, and 24 exactly once to produce the target 14.'
assert countdown_check('19 + 24 - 29 = 14',_demo)[0]
assert not countdown_check('29 - 19 + 24 = 14',_demo)[0]
assert not countdown_check('29-19=10 FINAL_ANSWER:14',_demo)[0]
print('Verification regression tests PASS')

### 5. Embedded verified 64-token BD3LM natural generations (descriptive reference only)


In [ ]:
REFERENCE_SHA256='441728d9510c6b491b1d5e6c811f612cda24bca07ec4617f314d6680166af3cd'
_ref_bytes=zlib.decompress(base64.b64decode("""eNrdXQ1z4zaS/Sssb642mcg+4osEvZe7yuclqWwqm+QqVZdMORQFWSxTpMIPO9rc/Pd7DZAUKUseeSxtPDOZeCQSaDSawOPrRgP+4+w6rs3Z5R9ncZZdrUpTmfI2rtMiv4qbWVpXZ5d12ZjJ2dRU9dU8/d3MrqomSUyFO/6FPzlLiuUqMyTDFTS/r7KijOuiXF/h3yQzVwsTz8qiWLY15vEyzdZnl2effCa++fvZ5CzNr6qYpLQV2nLLYmYyFJtl2fJ80Uz//R93JucX6vxT3CjP/Qv1yflXeYVmk/p8ls7nTQW9z6czgfK3/gU7a2VcVYsYcuQ0mYvY96dxYBKuZ1EyjxJhpomMVaxmic+SwOciQbUcJrg1V2lewxwmJ3tAAMwDC5wnRQ7L4JK1xRyXXI2mjLOreZxmDcx4dsnl5uq1yU1pzdreKM0qTsvOlFTc7y/WZRpnuBKgXGUyk9RoaCNX42oNWVWdJhCdxLcmrqHd5xvDe6s0K+q/eauymGZmeZ5kTYWeeNbKaX79Ny8vvFVRZGbmffvF5x49w7hMK6jnTU19Z0zuxWWySGs0Tq3+zVuYprRNellc1ed5s5xC4K0p03mKD6X5rUlR0LOj5gIGabs2b7KrrrPpdR7jgc7RO/MKzybO0zmGFY0+2HSeXttxmLRm+vmsRp+5giz6oPz2Q0hXrrNiClkvMTCzIrk5uxR8x8CC2a8qY2a4rVnEuuGQzv6kUXVVFU2ZdIOjaOqrurgxefus8bTqIiloyJfV7wwaY6gtpynG2fltcM59HjCfRWeb0YOukaGoc3xCP8XLbshcLePf3VixTV5B0jKlYZLELAxCxefTxJhERkkwlWEYxjxhOjZaG2mmzIfN+qqreJ0V8aztb5L4cqq5r9g84HMdSH8a8UQyrUI2U2Y+E/OZjoSQ07mcGsaD6Tw2wVz5ej5XgmuSW5tV2+faLFc0MzDIaNYzhXGRlBjQGPJNnaA96vY588+Z/pFFl1JdivDCD6WQ6kPfv/RpVFyvGhT80VRZ7P0oqQEaaU7mmcSjMVxHfM6CIJ7ChDoK51oxnzMlmdImFHFspmLO1VwoKTnTUoQBE5zNeETPDVMqWVxhrFcOB/gFYxf+h0nDhB2UZZxX86JcosCglLxQ0PTsFT2upCjtk/rjbBbXcWXsgyiavJ4Vd/nVHBNn4YZrixLjUWvHa1/8ErM+nbpyl0z76lL6/2dL0hCbo89BYAfXqqFmfiy8qshujVcv0qpDhIl3Z7wcbXh14Tl7e7F3C8EzDwAOfagTqBLXXgN4wieUt3O+8ng08fA3zmeeRmlM2GztFXliSBgamDX0ERXquLw2tcf4hfcjvg4EV4uiyWbAGkBPnqSVsdIM/r9L64X36xdfffvxN1cff/vDT59/f+n9h2v5P3+9+CX/Jf/SlOavABqPhtH5dH1O/3rxCi3HyYJ0wD+paXt8CaOsTHxzdVvGy6vr6dklu2ChzyOulRR+5GtuZx5Z5er1hj6zhZcrsuz/VObJdvneYJjmve0BwvViaQhmB9Y6zDaEC/Fd96rB/Bliy5FnBL2AGog+K24sKlN/YFlOr22M75sNR7BgT1MIusC+NAkYl9GEsYCriWTRREiMccYmCk+CoaWJwBfAiZpwaM6DKJyEMhITCfsCY9iEh/jA/GjCuT9h4QRYSlfxxX0SXLhvYhLoKJgIH7JIKAGkrc1Z0NYOSAATEBy4VgBYKKzDSSBUJK0sLaCo1BOmcCvSGh+DEAg6EZFQaBjlMeMkjSQpAo4G8UlLiCcJ1AkB2J4wKfATA8/XEyV14PrJ/NCZgUkZAr3dGzWJHeX44wyPF1cSomazNHFXnZ27O72pHf/CmwvlRuXzJsv6G/eezK7HZUfj1WYIOhmAsjuiiZUhCkRvsYtIB4pFkguf/kjxanIaiNMPIBwBgpcSIny3rheELkXWUFUPoGwnXDu/Lwk+fv3115Ut9ks+M3Ovmy6Dzr7/AQp6+NNN7I+8n5mceMyfeFy8dPcG0/Mjb/7L2R9t4Z/9l6+8/gsbfuEvX/1y5mqXbtqj3ng2/7ERa8v+kq9KkND3r4+JYvohEOs6SpjDxUEw9q6h2EEgZue3CN38Bg/gQAyOkSwIWcIO1TCnJ6EFEuAK7rBIT0LlT4KIMEr7gSYQ4RDCW0gD/oEqWWCKOlijL6r/ArzSk4gzW8liFlXihJ6SASNFKEIgjLZ1RKRxBQ91czXYeTVsrwbQkATL0Hcy9yAeoyb9KAIs8xCwKdArwQN9IITtRrDWzv9iCAtCyUQY8gh/QhWeCsJkeCoMs/zuqi2wC734xBMTT7XQ1c7bjzzh78AywE4LUtRy6qW5B357bd7PTP5+K/ODro2t2h9+BFJWdqV+Tl9+4H0IgfT3Q6+/+iF7eVRSFj5IymzXCXvUIWAm/HcMzYT/L4AzR+YCHxCyD8/CHr90z9b8HsYcISMYa0soK3uEcFoAnugK6RT4UNdHeV8DmyKfAbC4H06iSIEQ6o7MwUwoGnY3SYAM8BWVhPDt//TZqkp3mQXbVqtD2dgzgjIwMKGEH1g2djIk4/qU/mbVLCuvWVFJxX24n2l+PZrS8LEwnBxD8UeT+sJ7pHNIYAr+8uLFDzZoYpv51jXz4sWl99NG6WpTYKPHMWGMP0jL9vd5D5DBdO8YklGPTuxeikAQovnOXWwxiqALcBQAN0ae58btBGRwPvQ8XZWN80kOJijfo/1BR9ZQPCSsCjsoDnEtUopwkvtq0yvBW/92rOOBOBY9HxzTgUTvlbQ4FqhT4RgLT4lj8WzW4ZiQO2AMz3bjaek/EcdYcEwcYw/SsQc6vY+RyXeNkclTwxhjAmA1hDGCimgHhhFysXtu5gDGRjG0Pw3GrJqH0rHgOfExhQERhUFkfUt5KiBTT8KxeYpZcwCKqV0oBv8KL5njgthXM1pyna9tQ985jbdg7LU6HxPS1GsczD0G2MfM3jVAU0cENBmGBwEagcJOQAuHcbLo1GjGNWpRJacGmsHPXZj2UMfePj9TK4yjMATm4M+pQE2wU7KzdWqy2XBd05O+d7cwuWdQoaGZtAPtBC0BtPSFHcv3/NQt8j9A2pJBiVaVY+KbYA8B3AN93oNw8l1zPeXpPU8lgTiEIUyE3OFVRHQsIlhRhDU7wE5vrQuM2VtwfCdUM0aB/wfpG1NRMOZvbyHAweywmYuj8dN5oDw6DsYtY0w//KCMsAMSOIizqMdEk865eCCHo53ikJZkRWVlbOq+Sc7GmAkeNcoWPcjl3sww7xTWUY+ODnZCRSF/TRrHiLrRz/3xtPvJHFKzdmV0ZzKHBVuSJuyVtnVb842TNe7xwLcQ5qTE/PEjZeNsUTCGuetqqW9eB2620GVdxml+qUTwNDybAs9uPJq5wyVPL80pcLWE8pTDSsmDdjY5YYYAxMIJXML0Np01KLe+8L4xNRAHY7ys3ZwjidfpLRhemlO2nkOFHmo+S5eYaJR66hVz53maVRbnNXExgj77hdZoYS1vbjD5p+v+M1COk5R/NCiU1muS8Z2J86ZGT6H1D781cWm8L1AU8nYimtYB8IyHIuIq1FuQtm3mIYr9Pa5rbxETvK5sk05V2z/3BtintOcxr3KazYuiJqVXrdJJnOOtcmNQ2LuGmtXgJup9NffSGiB4g1cLv1+CHqarvX3LmzY1pSJbNundXN+/U5ksq+zi83ugm4vizls2eMSzAk3hMVixc/Tds6Oo7+l/PYPIHgNTO6IvLIBmMlKDBViGwhEDAobQywKaE8I0wM8CF3SIiERSwgazmMZoslL+moW8IICYCELvQRij1A7BVAvMIqLFVR62HI8wFXRIu0Xi1gNX9DeEzSZKiR1XA1p/CPsmWCgkt01Y8SGnlBJtwT2A6gG63nrPb18WCSVCqzDSmqCUiSciKaj+02J6ID5JkxE5tGSlqMELk6KqO2yrz6tFWmKiTovmemFRgS4nTVUXMKcje4NqBAeUrduJnVGNZZPV6Spbj71k20Qvft0ieQreREDY3rnwvsqd9glsNNlXHfcDCxU7hdDt97h/8WiCuG3qIZp+HS8BNNNmTU33ihAcKf/foNwc6AfjrN2LxxqHdPC8L3ukAgckdeP1M8Ck4IiAxHzZxrngHeGHFj1ecEUcLAgwyf2QMiqUI2eADN5Rsk2tQAaWlrFAtWW1pC0Mvf9rxfYiO2ECalm8oFtCBIFle6HSjpMxaRdh9wjpcaullbvlUSk8i86bFocCkf+MFh1UFEUyUJFFopA/EYrg5odHh6I7kxLqtGBUwnVNc0KRpEA3QAWWcRKXBXGxeE6cYBUnNx3KTEvih9P4uupI3hZttAxxEF57bfsbiZcvXtjEtnOa4mBFxIzksMX27uc093HBCQK78uBy4nHthCIQu1DDswhYGD2ERGTpERIVC2qVKBbj900zcQhkNSDb9Dp4X9qu5tZqRJU9A46XDfHV8qjOxGmOPtJOJ/QYDHfT2wmeTDxbW/CbmQzcuQSx/olEW6ulZg7vODPxbRtguEnrhO6Sr5wmNw61F0UOgKfRkaV1nVlz4wY4cW0sf69dZVrzKe2GkBofLWVEzfvPCFD7U0tqtx/ohPrirDA5YHA9A3iW6lTwLAPt9/AcBeTYCqnCCRSndDxA74ST78s4wbiiaJ5DTh3Spgi9zSJHbNTSz20GKUMA7R4FhmKJrxJV5Sqwfjsn8qdwtYt6DsoSg+zKaUwkqA+UlhEs3PFNqX2o1hY8EK/Vc1pLiQSIih/aJeInu+AhD46I1llRVc6F/DQuY8vEvjZluYZHnWXOFUvtNFsP1yjS0lti6q6tS9/AF7YJxK3TDu+Oyiw9CNWWUbn53VgaaTnem+N6vKTQH8l0CsS3cZrF6OQI2HcXfqJTDsMPwZuasU+s6+C2amTQifd1nJt6MjDsAmBK0CYv1WVwQX72/t5Zu3cV3gvVZNfT2zw4aurED+7PB9ToBHgqQz+ycUtAEoGd3bsRwhcOAEjAT7jXYJXaD9u1D01JyoRONpslAGZqSRhFFaVS1LJPPrDutqkBJOUGKpkWwnHTpwGwsM1DqtWFK9+/B709LR+VfQsTnyljnQtf2vWa7USbN3C/5dPc75kBa12Ol2xpwiSLNJuBZXnv//DBB0RWfsDUKgtgndu7fuH9t41QWshFTYrqCc/uy6c52F6p0t+9OiU3FV+Wcb72ZjF58bSEEVdtKasPue1jTNihy4OevDsTYO0Fw3UcCsqti4bEP9H3liPf21njkD6SWhYdbdCiJOKarW3FelEaY+9PXPCQ6vZ97aKIhJrPgP5xdjS40lyJLd8XTjXmM/xnf6LxH+2XANWAiBBkyS53gHEQekAyPHK7MoOBjsfXLS9Dpt3XJfobMpSENgKASHgU0HaLSBMnw9euUKsbufkbFB3rtdfppwb7aCIbL88wq2akSfKBIPWMsgG1FtruNbObM5R8MtULj++YbyjGvChm3neEY44zEB+YQteZV+SDtZT+MJR93ni3iGOXaAAk9O+hzjmmbXJj3NrpIl5OG8yacuCkf+ruX3os8FZQfFa1NwaF96U3P4LZhSO3/It2Q55DbPiXTY7uT1pbpVXHiHqdqCdT0Kpz4P24R9m8TVh0Vsf9jeIUUSSsvS4sKDtCxsHP4CNv5C6KGqa+tiR9V20M0WIsooLTXcLKVTozLbe3z9davdNpSFe7JroQJ+HpRgE7TKwAxwWfwzJMdCqfesOSaFMZ3ZSWA1qOh3vAWB1t1lngAfMHOJxt0S7aEN61y86HudQsgH1aXNaoIrS+x+2EUPbbMNE7EITOtLOuI4DD6m91qrWQAd5KnB0HXMm+JwTXVXEHrGgqt47ydVGZ1WKzqHkdzzCQHwOoFwd6xbva3cBpq8cqXrvQ2ri0C6bNy/Ta2FOm0F2A2cLF4YhzObp2r+JTQqL0GMYhUauhW+weKxN7dzEB8MLYfwhLU2KqDS0vEdxSHJT2vmDweUVPsjGAqob0djrjyiYSSvhVXeyPGDgJQ6t1m7VpmF2v95pul81a2W7haCPp/jMYdrMPEIx7ZjO+XM27oqwX3nu05J9SihIUrxcTr76jtbS7ffq7ututWbmTh4b2a7WvTD+6n8FJD7DKyV8VXFDupdTCojxTjJa/wcS1H0yUEKF63BvChmxf7/LvanX7BdFeRefadKhRHRv4oEAr9VTT+WK0gCboeBti//cqvI1JnhoGgA9qI6+heiofx8A7+itj4IjDBDmIHmZa66Q3N02GeY8Zed2kGNOVw7nbtMjS3OFW3kcBmjbV8V4G1EPhik4UvV2G+VSYy52yXcRgj7qW6bfrZa2+Fr1lVwCyz5/yhiCrj9h5GdOBYn/t6Dk1xgemkmNjxV5iyhqCdvT6Ia3dPSfK3gq2bjkh27VG4dw3f7Y2QOx31QZxju6+DXPcN8XziHmcCnL7IELkB/ZcmjYXQAaUe4Q/k8BlCABoo/6kMBkIOvFC+DZNwbeZ7H30IrIZC8O0qd2hlo0Qi8++Hwxyryx2wzByn6qO9XO3wDVQlw7W6YMxkStrU6oOPZpCPqf8qEBifGgLtpxtgW11Gy9XrwNbW6id9gkc1fMoEEcE3M10XJlilRE7KjChqpSynajAJ41NQoCTDZKTxcSfUshuWXlc1Z5W3gz054L27hDozgrX+IMR1j2t3W+ild7xwJ0S7oi3tvn39/UtOg/CACiaisQNwrqPD5PseiJDMP7YJm8Y95aIiUWWNUxh4swtPXXaVsbs1vjC5WFMvO+BqSajXIMytwYCcZXKGawoKYtsaR7Xb8JizaPOeG/0oDcBkV6Rh+V0CbmzZ5EbFmp5AizusY1xBay0K1pR5MAPwIuyUghikDp0OanglGG3EkY/tVv5sruKCIQD4KdsOfCOkPGOZrYa2JKtttG7lyAiEfa7A4a6dmEdERBvpsSEYVz6QDTWzyjrIGJKhJFyh55FKjgGHLMnnqexj4Wuirry3p+iB9UHrdPuslHtmbuz7ez+TcV5Ru5n68FvyYMOLiuqBY17tTy7mWDjdt8vQJmokDVgxzOTF/feJlvFafXq1/baFa5d0bWPPPnohbRd9h+Cr6WhJNxyUL/XAziW1hfDBDd/dNOS04thrBfmonK1rUFRhCx7BujF/BOunYEICrd4Jm36u7arY33eqmScb+XU92ASUUJAm/K6LQ8uNHig7DFmu1YYhrvwyd60Kf6C0ghGFDPUfiAfLG9DCnR2LflUAKyAjnzE5OebE9eit5FQcnqWdAZlBEA+DqHU7CQIdl3Siforytc0bo7RYgm6T/tx7q+pbSrClH21Lg1+23ncKfaxseO//OUv3Ukb3idUh6q0IeXvDS0E2kZoLxTzSa2nwxWMPYQrJ3bU26oAqbMLXmMDxg6Ghr11jnULaHFrIxiHqT1m7+HtNaKfAdCpE+JcwAilfEXbjThtzvFbLJE+lzuWtAYL9bTXclCxzakfu7d7JT8+5inbBH4SoQJqjc7QpbWsQTg08IVwm4tco27v0mY3PD98bxHzn9M+TRHSQ7RsTYTiGFgX+g9h3fdrUKxqZWiVN6BVFHADYJT1v8iZM/l1llYLt1N6x/0E7pKpyH8bomIfp7KT2tWixE7Xzi75blnbuIDWjgb6+GbVTNG50clmbvG6baPzHDvBdrHtNWW7Tgwyn+wmqS702W4fHZzGa0VctW08HSLxjEaMjhLwyXf0jvZ4vtzxRLocKSv/jZ7Kc0isPwg0pZZE6ULWRtk4HD5aFCHMCzQdZKspoXOwD318WwWWDpJvyaSDV5ujZPOitC9ccUqLcm1syWU+hQcDgial70ndhCEFpzWbccSxExxRPpQTp3z5UCEndrzf3e6CJ69359m+wqfSdD5vxA5OC31OmzIVRpCv3VYoLo6Cm3jPnCToOC2KG0d7lsVtinl157IU7R6cjvzl7uAzqtktUyem/5VNlh3eFf1JRAS3d30kMq27tKtPH1Rh1O6DkcstRe9i2v2zw2G+L5e2yx8h3IhnsZVz3zJAkMnBhqbessHgaqt8S/b+6iVl/E8Q4jTL8DNZFEWGixUGsXHrNpTvak3CZL+FyokgyV3fWdTnxdtWx/jqrln0bOtai3Si3ymuuSMgGNDZ24SlER0JbsGNERWUkdvKTis27qA02jkahJSqJPtTkfToBCMLsx0+2gih4MP0KfoFVbuaH7W6J6I4Vk8JOsP8HnG+J8xtdn8Lf/8BZhoGiLLkUupjYCQe0EMYSRyq3YtutzCOtnMz1cam5kXZbgwH5lGKSV65hJb21CEq6r3wlPeRx7i6eDrbgtZDNPmsKTvI+7i8Qeuxm+o/mt8pzdzQ+q9ddUCB7+N/FuU0pi2PbV+qRbHCzJ9BT48UJWfbZJnbqNltK78Y7WTcZZPnwKQOOw8o1GqzJZzt31Ud2DMiKCeFh2G/X1tRdiPn5IsKd2TPqLzYHC/Rx6yC7pygg1NP+LMKwGs4woFv10P1UZgJSN7DzKSdQZaa/NaYqrZ5ywdnoKyKlPblftPcwDdICorWtG9POw9GgaxBCMulCnfRq3viKCxuy7yWtnxm5vdDa534ymMqdKfw7Cy33RpqHCGEJUcH8VjLXMf0CxvBEoKuTRjJTmnXLnSJrcG67SurLF5TBRV2nbFFMOhuU1dyGIu3EtsTJuxDGD6D55DdAfs9IsPDznnHITaZF69N8BDKB25ISe4W+KLeBJe0H42jVoOguBKbox/GogitwDsexzCY7lI6ttqgcFaXSu3TbwbYHIOzo85YAdR9C92sMJToinDhqe3c6zcDMxmcJrWjqRbnTXuAWFI2uY362kXF/8UcjSm+ks62cuf2xJZ6WTtiSRvRxcZla2+7LRxdZesNdMUPybQ7uDNHcLDkaE/zwEKeUOPmxUYBANL1euklWUybxwubbPFZfItKVFHpLQNMm9rToZdhkI06kXv/dO1tR6l2m84K7zUsnsM+FH5KZ0pKIkIB9HBpbHS4FsiRW6gUmpOPYn8JHGe7Mtt2hZQ2EndEkjrxm9Osu/s2zrWpa4Nag/KvS4h7RI8OhEYHdc/Gu2L02/44haDEq5ev/h+NlXWD"""))
assert hashlib.sha256(_ref_bytes).hexdigest()==REFERENCE_SHA256
REFERENCE_DATA=json.loads(_ref_bytes.decode('utf-8'))
REFERENCE=REFERENCE_DATA['records']
assert len(REFERENCE)==24 and all(x['status']=='ok' for x in REFERENCE)
assert REFERENCE_DATA['manifest']['config']['source_payload_sha']==PAYLOAD_SHA
assert REFERENCE_DATA['manifest']['config']['model_sha']=='4bcf3a00ba6ec28d9cf9c3ebc4a5a5dc01c6023c'
assert {x['problem_id'] for x in REFERENCE}=={s['problem_id'] for s in SOURCES}
print('BD3LM 64-token natural baseline:',sum(int(r['task_success']) for r in REFERENCE),'/24')
print('Reference hash verified:',REFERENCE_SHA256)


### 6. Model loader, native BD3LM sampler, run-signature checks, and append-only checkpointing


In [ ]:
PROTOCOL='rsx23-bd3lm-fixed32-per-block-suffix-repair-v1'
MODELS={
    'MDLM': dict(
        repo='dllm-hub/Qwen2.5-Coder-0.5B-Instruct-diffusion-mdlm-v0.1',
        sha='a284e895a6248256baf2f60502e54aba61b24c1a',
        actions=['tail25','tail50','tail75','global'],
        sampler_cls=MDLMSampler,config_cls=MDLMSamplerConfig,
    ),
    'BD3LM': dict(
        repo='dllm-hub/Qwen2.5-Coder-0.5B-Instruct-diffusion-bd3lm-v0.1',
        sha='4bcf3a00ba6ec28d9cf9c3ebc4a5a5dc01c6023c',
        actions=['tail25','tail50','tail75','global'],
        sampler_cls=BD3LMSampler,config_cls=BD3LMSamplerConfig,
    ),
}
OUT_LEN=128; BLOCK=32; STEPS=128; TEMPERATURE=.15
GEN_SEED=38191; REPAIR_SEEDS=[38192,38193]; SELECT_MAX=8
PROMPT_SUFFIX='\nReturn a concise answer. For word problems finish with FINAL_ANSWER: <number>. For Countdown give a valid arithmetic expression using each supplied number once.'
CTX=None
SRC_MAP={s['problem_id']:s for s in SOURCES}
assert len(SRC_MAP)==24

def normalized_json(x):
    if isinstance(x,Path):return str(x)
    if isinstance(x,torch.Tensor):return x.detach().cpu().tolist()
    if hasattr(x,'item'):
        try:return x.item()
        except Exception:pass
    raise TypeError('Nonserializable '+repr(type(x)))

def atomic_json(path,value):
    path=Path(path);p=path.with_suffix(path.suffix+'.tmp')
    with p.open('w',encoding='utf-8') as f:
        json.dump(value,f,indent=2,ensure_ascii=False,default=normalized_json,allow_nan=False)
        f.flush();os.fsync(f.fileno())
    os.replace(p,path)

def append_record(path,row):
    with open(path,'a',encoding='utf-8') as f:
        f.write(json.dumps(row,ensure_ascii=False,default=normalized_json,allow_nan=False)+'\n')
        f.flush();os.fsync(f.fileno())

def completed_map(path,signature):
    records={}
    if not Path(path).exists():return records
    with open(path,encoding='utf-8') as f:
        for lineno,line in enumerate(f,1):
            if not line.strip():continue
            try:r=json.loads(line)
            except Exception as ex:raise ValueError(f'Malformed checkpoint JSONL {path}:{lineno}; backup before editing') from ex
            if r['signature']!=signature:raise ValueError('Checkpoint run-signature mismatch: '+str(path))
            if r['id'] in records:raise ValueError('Duplicate trial ID in checkpoint: '+str(r['id']))
            records[r['id']]=r
    return records

def model_dir(family):
    f=DRIVE/family
    f.mkdir(parents=True,exist_ok=True)
    return f

def config_signature(family):
    assert family=='BD3LM', 'Notebook 23 runs BD3LM only.'
    m=MODELS[family]
    frozen=dict(protocol=PROTOCOL,family=family,model_id=m['repo'],model_sha=m['sha'],
                source_commit=SOURCE_COMMIT,source_payload_sha=PAYLOAD_SHA,
                n_sources=len(SOURCES),out_tokens=OUT_LEN,block=BLOCK,configured_steps_per_block=32,
                temperature=TEMPERATURE,gen_seed=GEN_SEED,repair_seeds=REPAIR_SEEDS,
                select_max=SELECT_MAX,actions=['tail25','tail50','tail75','global'],budgets=[128,256],bd3lm64_reference_sha=REFERENCE_SHA256)
    return hashlib.sha256(json.dumps(frozen,sort_keys=True).encode()).hexdigest(),frozen

def files_for(family):
    folder=model_dir(family)
    return {'folder':folder,'manifest':folder/'run_manifest.json',
            'natural':folder/'generation_trials.jsonl',
            'repair':folder/'repair_trials.jsonl',
            'cohort':folder/'frozen_failure_ids.json',
            'errors':folder/'errors.jsonl'}

def init_family(family):
    paths=files_for(family);sig,cfg=config_signature(family)
    if paths['manifest'].exists():
        old=json.loads(paths['manifest'].read_text())
        if old['signature']!=sig:raise RuntimeError('Different protocol found for '+family+'; do not overwrite existing run. Rename Drive folder or restore previous notebook.')
    else:
        atomic_json(paths['manifest'],dict(signature=sig,config=cfg,
            created_utc=datetime.now(timezone.utc).isoformat(),gpu=GPU,torch_version=torch.__version__,transformers_version=transformers.__version__))
    return paths,sig

def unload_model():
    global CTX
    if CTX is not None:
        print('Unloading',CTX['family'])
        CTX['sampler']=None
        CTX['model']=None
        CTX['tokenizer']=None
        CTX=None
    gc.collect()
    torch.cuda.empty_cache()
    gc.collect()

def load_family(family):
    global CTX
    assert family in MODELS
    if CTX and CTX['family']==family:
        print(family,'already loaded');return
    unload_model()
    m=MODELS[family]; paths,sig=init_family(family)
    print('Loading',family,'public Hugging Face checkpoint (no token needed)',m['repo'])
    # HF cache-first; otherwise pinned full-snapshot download; simple retry.
    try:
        local=snapshot_download(repo_id=m['repo'],revision=m['sha'],local_files_only=True)
    except Exception:
        for trial in range(3):
            try:
                local=snapshot_download(repo_id=m['repo'],revision=m['sha'],max_workers=4)
                break
            except Exception:
                if trial==2:raise
                time.sleep(2**trial)
    local=Path(local)
    assert (local/'config.json').exists() and (local/'model.safetensors').exists()
    try:
        tokenizer=AutoTokenizer.from_pretrained(str(local),local_files_only=True,trust_remote_code=False,use_fast=True)
        if tokenizer.pad_token_id is None:tokenizer.pad_token=tokenizer.eos_token
        if tokenizer.bos_token_id is None:tokenizer.bos_token=tokenizer.pad_token
        if tokenizer.mask_token_id is None:
            tokenizer.mask_token='<|mask|>'
        mask_id=int(tokenizer.mask_token_id)
        pad_id=int(tokenizer.pad_token_id)
        eos_id=int(tokenizer.eos_token_id)
        config=A2DQwen2Config.from_pretrained(str(local),local_files_only=True)
        if config.model_type!='a2d-qwen2':raise ValueError('Wrong checkpoint architecture: '+str(config.model_type))
        model=A2DQwen2LMHeadModel.from_pretrained(
            str(local),config=config,dtype=torch.float16,
            local_files_only=True,low_cpu_mem_usage=True,attn_implementation='sdpa',
        ).to('cuda').eval()
        nrows=int(model.get_input_embeddings().weight.shape[0])
        assert 0<=mask_id<nrows and mask_id not in [pad_id,eos_id],('Mask token missing from trained vocabulary',mask_id,nrows)
        sampler=m['sampler_cls'](model=model,tokenizer=tokenizer)
        CTX=dict(family=family,model=model,tokenizer=tokenizer,sampler=sampler,
                 mask=mask_id,eos=eos_id,pad=pad_id,paths=paths,signature=sig,
                 model_sha=m['sha'],model_id=m['repo'],classes=m)
        print('MODEL LOADED',family,'GPU allocated GB',round(torch.cuda.memory_allocated()/2**30,3),'mask id',mask_id)
    except Exception as exc:
        append_record(paths['errors'],dict(stage='model_load',signature=sig,error=repr(exc),traceback=traceback.format_exc(),timestamp=datetime.now(timezone.utc).isoformat()))
        raise

def prepare_ids(s):
    t=CTX['tokenizer']
    msg=[{'role':'system','content':'You solve quantitative reasoning tasks carefully.'},
         {'role':'user','content':s['prompt']+PROMPT_SUFFIX}]
    ids=t.apply_chat_template(msg,tokenize=True,add_generation_prompt=True,enable_thinking=False)
    if isinstance(ids,dict):ids=ids['input_ids']
    ids=[int(i) for i in ids]
    assert ids and CTX['mask'] not in ids
    return ids

def visible_answer_tokens(token_ids):
    result=[]
    for i in token_ids:
        if int(i) in [CTX['eos'],CTX['pad']]:break
        result.append(int(i))
    return result

def decode(token_ids):
    return CTX['tokenizer'].decode([int(i) for i in token_ids],skip_special_tokens=True).strip()

def sampler_config(answer_budget):
    # Fixed 32 steps for EACH 32-token block, NOT a fixed overall NFE.
    requested_steps=32*math.ceil(int(answer_budget)/BLOCK)
    a=dict(max_new_tokens=int(answer_budget),steps=requested_steps,block_size=BLOCK,
           temperature=TEMPERATURE,remasking='low_confidence',return_dict=True,
           right_shift_logits=False)
    if CTX['family']=='BD3LM':a['steps_per_block']=32
    return CTX['classes']['config_cls'](**a)

class CountCalls:
    def __init__(self,module):self.module=module;self.count=0;self.h=None
    def cb(self,*args):self.count+=1
    def __enter__(self):self.count=0;self.h=self.module.register_forward_pre_hook(self.cb);return self
    def __exit__(self,*args):self.h.remove();self.h=None

def set_seed(seed):
    torch.manual_seed(int(seed));torch.cuda.manual_seed_all(int(seed));random.seed(int(seed))

@torch.no_grad()
def real_generate(prompt,answer_budget,seed):
    set_seed(seed)
    t=time.perf_counter()
    torch.cuda.reset_peak_memory_stats()
    with CountCalls(CTX['model']) as calls:
        result=CTX['sampler'].sample([list(prompt)],sampler_config(answer_budget),return_dict=True)
    raw=result.sequences[0].detach().cpu().tolist()
    if CTX['family']=='MDLM':
        assert raw[:len(prompt)]==list(prompt), 'MDLM changed observed prompt'
        answer=raw[len(prompt):]
    else:
        offset=math.ceil(len(prompt)/BLOCK)*BLOCK
        assert raw[offset-len(prompt):offset]==list(prompt),'BD3LM changed observed prompt'
        answer=raw[offset:]
    assert not any(t==CTX['mask'] for t in answer), 'Unresolved generation mask(s)'
    assert len(answer)<=answer_budget
    return dict(token_ids=visible_answer_tokens(answer),nfe=int(calls.count),
                raw_generated_tokens=len(answer),wall_seconds=float(time.perf_counter()-t),
                peak_vram_gb=float(torch.cuda.max_memory_allocated()/2**30))


### 7. Real CUDA smoke test: BD3LM native 32-token sampling (fails early if the loader is incompatible)


In [ ]:
load_family('BD3LM')
source_sample=SOURCES[0]
prompt=prepare_ids(source_sample)
smoke=real_generate(prompt,32,45001)
assert smoke['nfe']>0
assert len(smoke['token_ids'])<=32
print('BD3LM real-model generation smoke test PASS; NFE=',smoke['nfe'], 'text=',repr(decode(smoke['token_ids'])[:180]))


### 8. Short, persistent BD3LM generation runner (32 requested steps per 32-token block)


In [ ]:
BUDGETS=(128,256)
assert CTX is not None and CTX['family']=='BD3LM'

def generation_trial(s,budget):
    ret=real_generate(prepare_ids(s),int(budget),GEN_SEED)
    decoded=decode(ret['token_ids'])
    v=verify(decoded,s)
    return dict(id=f"{s['problem_id']}|fixed32x32|{budget}|{GEN_SEED}",problem_id=s['problem_id'],
       dataset=s['dataset'],target=float(s['target']),prompt=s['prompt'],budget=int(budget),
       generation_seed=GEN_SEED,output=decoded,verification=v,
       task_success=bool(v['task_success']),strict_success=bool(v['strict_success']),
       numeric_success=bool(v['numeric_success']), visible_answer_tokens=len(ret['token_ids']),
       step_policy='32_per_32token_block',configured_total_steps=32*math.ceil(budget/BLOCK),
       budget_exhaustion_proxy=bool(ret['raw_generated_tokens']==budget and len(ret['token_ids'])>=budget-2),**ret)

def gen_progress():
    done=completed_map(CTX['paths']['natural'],CTX['signature'])
    for budget in BUDGETS:
        ready=sum(int(x['budget']==budget and x['status']=='ok') for x in done.values())
        print(f'Controlled generation {budget}: {ready}/24 completed')

def generation_batch(budget,start,n=4,max_minutes=8):
    assert CTX is not None and CTX['family']=='BD3LM'
    assert int(budget) in BUDGETS and 0<=start<24 and 1<=n<=4 and start+n<=24
    path=CTX['paths']['natural'];sig=CTX['signature'];done=completed_map(path,sig)
    began=time.monotonic()
    for s in SOURCES[start:start+n]:
        tid=f"{s['problem_id']}|fixed32x32|{budget}|{GEN_SEED}"
        if tid in done:
            assert done[tid]['status']=='ok'
            print('SKIP',tid);continue
        if time.monotonic()-began>max_minutes*60:
            print('Reached soft cutoff. Rerun this batch to resume.');break
        try:
            r=generation_trial(s,budget);r.update(signature=sig,status='ok')
            append_record(path,r)
            print('SAVED',tid,'task=',r['task_success'],'NFEs=',r['nfe'])
        except Exception as exc:
            append_record(CTX['paths']['errors'],dict(stage='generation',id=tid,error=repr(exc),
                  traceback=traceback.format_exc(),utc=datetime.now(timezone.utc).isoformat()))
            raise RuntimeError('Generation failed. Existing Drive checkpoints are safe. Check errors.jsonl.') from exc
    gen_progress()


### 9. 128-token BD3LM: 6 short batches × 4 generations


In [ ]:
generation_batch(128,0,n=4,max_minutes=8)

In [ ]:
generation_batch(128,4,n=4,max_minutes=8)

In [ ]:
generation_batch(128,8,n=4,max_minutes=8)

In [ ]:
generation_batch(128,12,n=4,max_minutes=8)

In [ ]:
generation_batch(128,16,n=4,max_minutes=8)

In [ ]:
generation_batch(128,20,n=4,max_minutes=8)

### 10. 256-token BD3LM: 6 short batches × 4 generations


In [ ]:
generation_batch(256,0,n=4,max_minutes=8)

In [ ]:
generation_batch(256,4,n=4,max_minutes=8)

In [ ]:
generation_batch(256,8,n=4,max_minutes=8)

In [ ]:
generation_batch(256,12,n=4,max_minutes=8)

In [ ]:
generation_batch(256,16,n=4,max_minutes=8)

In [ ]:
generation_batch(256,20,n=4,max_minutes=8)

### 10a. Export all 48 generation trials before attempting any repairs


In [ ]:
# Export the entire generation stage separately before proceeding to repairs.
assert CTX and CTX['family']=='BD3LM'
_checkpoint=completed_map(CTX['paths']['natural'],CTX['signature'])
_expected={f"{s['problem_id']}|fixed32x32|{b}|{GEN_SEED}" for s in SOURCES for b in BUDGETS}
assert set(_checkpoint)==_expected, f'Generation stage incomplete: {len(_expected-set(_checkpoint))} missing'
assert all(x['status']=='ok' for x in _checkpoint.values())
_export_zip=Path('/content/RepairScopeX_NB23_BD3LM_ConstantPerBlock_Generation_only.zip')
with zipfile.ZipFile(_export_zip,'w',compression=zipfile.ZIP_DEFLATED) as _z:
    _z.write(CTX['paths']['manifest'],CTX['paths']['manifest'].name)
    _z.write(CTX['paths']['natural'],CTX['paths']['natural'].name)
print('Generation-only backup verified:',_export_zip)
from google.colab import files
files.download(str(_export_zip))


### 11. Freeze exactly eight native failures (corrected metadata-only cohort mismatch)

This cell checks a SHA-256 fingerprint of **scientific trial content**, independent of runtime telemetry.
It only accepts the two generation files whose differences were audited and never deletes the saved cohort.
If the scientific results, cohort IDs, or previously saved repair records disagree, it stops.


In [ ]:
# Notebook 23 — corrected Section 11 (frozen failure cohort)
# Replace the ENTIRE old Section 11 code cell with this code and execute it.
# Run the existing setup, model loader, and generation-batch definition cells first.
# Generation does not need to be rerun when its 48 checkpoints already exist.

from collections import defaultdict

# The two supplied archives were independently compared. They differ ONLY
# in wall_seconds / peak_vram_gb for 13 of 48 records.
# These are real-data audit constants, not fabricated reference measurements.
RSX23_AUDITED_RAW_SHA256 = {
    'f41a05d5995bff86a57f7b100d04d0d5d794757b0b92b674e6dcd24130addb5f',
    '019dd91e80e0bc4d910f98c7c7a7a84e0cfe93ccb921ad2fbfc4b3f94dd43c8a',
}
RSX23_AUDITED_SCIENTIFIC_SHA256 = (
    'e3a275aff724df5ce1d3870eb2cbe636c6321b739b22c4f31a0c59efa19099e6'
)
RSX23_TELEMETRY_FIELDS = {'wall_seconds', 'peak_vram_gb'}


def scientific_generation_sha256(rows):
    """Canonical hash of full records minus run-dependent performance telemetry."""
    stable = [
        {k: v for k, v in r.items() if k not in RSX23_TELEMETRY_FIELDS}
        for r in sorted(rows, key=lambda r: r['id'])
    ]
    blob = json.dumps(
        stable, ensure_ascii=False, sort_keys=True, separators=(',', ':'),
        allow_nan=False,
    ).encode('utf-8')
    return hashlib.sha256(blob).hexdigest()


def check_preexisting_repairs(paths, sig, selected):
    """Never accept a changed frozen cohort if repair records contradict it."""
    repaired = completed_map(paths['repair'], sig)
    chosen = {r['problem_id']: r for r in selected}
    for rid, record in repaired.items():
        pid = record.get('problem_id')
        if pid not in chosen:
            raise RuntimeError(
                'Existing repair record belongs to a different failure cohort: ' + str(rid)
            )
        if record.get('original_output') != chosen[pid]['output']:
            raise RuntimeError(
                'Existing repair uses a different original failed output: ' + str(rid)
            )
        if record.get('status') != 'ok':
            raise RuntimeError('Failed repair checkpoint detected: ' + str(rid))
    return len(repaired)


def choose_failures():
    assert CTX is not None and CTX['family'] == 'BD3LM'
    p = CTX['paths']
    sig = CTX['signature']
    gens = completed_map(p['natural'], sig)
    expected = {
        f"{s['problem_id']}|fixed32x32|{b}|{GEN_SEED}"
        for s in SOURCES for b in BUDGETS
    }
    if set(gens) != expected or len(gens) != 48:
        missing = sorted(expected - set(gens))
        unexpected = sorted(set(gens) - expected)
        raise RuntimeError(
            f'Incomplete/extra generation checkpoints. Missing={missing[:5]}, '
            f'unexpected={unexpected[:5]}'
        )
    if not all(r['status'] == 'ok' for r in gens.values()):
        raise RuntimeError('Generation checkpoint contains non-success records.')

    # Candidate selection remains EXACTLY the original, pre-registered algorithm.
    pool = defaultdict(list)
    for r in gens.values():
        if r['budget'] == 128 and not r['task_success'] and len(r['token_ids']) >= 16:
            assert not verify(r['output'], SRC_MAP[r['problem_id']])['task_success']
            pool[r['dataset']].append(r)

    desired = [('countdown_fresh', 3), ('gsm8k', 3), ('svamp', 2)]
    if not all(len(pool[dataset]) >= n for dataset, n in desired):
        raise RuntimeError('Not enough genuine 128-token failures for fixed cohort.')

    selected = []
    for dataset, n in desired:
        sorted_pool = sorted(
            pool[dataset],
            key=lambda r: hashlib.sha256(
                ('rsx23cohort:' + r['problem_id']).encode()
            ).hexdigest(),
        )
        selected.extend(sorted_pool[:n])

    ids = [r['problem_id'] for r in selected]
    if len(ids) != 8 or len(set(ids)) != 8:
        raise RuntimeError('Selected cohort must contain 8 different original problems.')

    raw_sha = hashlib.sha256(p['natural'].read_bytes()).hexdigest()
    stable_sha = scientific_generation_sha256(list(gens.values()))
    core = {
        'generation_signature': sig,
        'ids': ids,
        'budget': 128,
        'dataset_counts': dict(Counter(r['dataset'] for r in selected)),
    }
    current = {
        'source_generation_sha256': raw_sha,
        **core,
        'source_generation_canonical_sha256': stable_sha,
    }

    # This notebook is specific to the uploaded, independently audited 48-trial
    # data. A new scientific fingerprint must NOT be silently accepted.
    if stable_sha != RSX23_AUDITED_SCIENTIFIC_SHA256:
        raise RuntimeError(
            'Scientific generation fingerprint differs from both uploaded archives. '
            'Do not overwrite the frozen cohort. Current fingerprint: ' + stable_sha
        )

    repairs_saved = check_preexisting_repairs(p, sig, selected)

    if p['cohort'].exists():
        old = json.loads(p['cohort'].read_text(encoding='utf-8'))
        changed = {
            k: (old.get(k), value) for k, value in core.items()
            if old.get(k) != value
        }
        if changed:
            raise RuntimeError(
                'Frozen cohort SCIENTIFIC metadata differs. Check the existing '
                'frozen_failure_ids.json before changing the experiment: '
                + repr(changed)
            )

        old_stable_sha = old.get('source_generation_canonical_sha256')
        if old_stable_sha is not None and old_stable_sha != stable_sha:
            raise RuntimeError('Frozen cohort scientific fingerprint changed.')

        old_raw_sha = old.get('source_generation_sha256')
        if old_raw_sha != raw_sha:
            # An old v1 cohort has no canonical fingerprint, so only migrate
            # the TWO RAW HASHES whose trial-level equivalence was verified.
            independently_verified = (
                old_raw_sha in RSX23_AUDITED_RAW_SHA256
                and raw_sha in RSX23_AUDITED_RAW_SHA256
                and stable_sha == RSX23_AUDITED_SCIENTIFIC_SHA256
            )
            previously_stable_verified = old_stable_sha == stable_sha
            if not (independently_verified or previously_stable_verified):
                print('Saved cohort raw SHA:', old_raw_sha)
                print('Current generations raw SHA:', raw_sha)
                print('Current scientific SHA:', stable_sha)
                raise RuntimeError(
                    'Unverified cohort source-file difference. Files are NOT '
                    'deleted or modified. Export Drive frozen_failure_ids.json '
                    'and generation_trials.jsonl for a manual audit.'
                )

            # Keep the ORIGINAL cohort file untouched; record the verified
            # telemetry-only reconciliation in a separate audit artifact.
            audit_path = p['folder'] / 'generation_cohort_reconciliation_v2.json'
            atomic_json(audit_path, {
                'status': 'verified_telemetry_only_mismatch',
                'existing_frozen_cohort_source_sha256': old_raw_sha,
                'current_generation_source_sha256': raw_sha,
                'canonical_scientific_sha256': stable_sha,
                'ignored_telemetry_keys': sorted(RSX23_TELEMETRY_FIELDS),
                'selected_problem_ids': ids,
                'existing_completed_repair_records': repairs_saved,
                'note': 'Original frozen_failure_ids.json was NOT changed.',
            })
            print('VERIFIED: mismatch affects runtime telemetry only.')
            print('Preserved existing frozen cohort:', p['cohort'])
            print('Wrote separate reconciliation audit:', audit_path)
        else:
            print('Existing frozen cohort is consistent; no changes needed.')
    else:
        atomic_json(p['cohort'], current)
        print('Created frozen failure cohort with scientific checksum:', p['cohort'])

    print('Selected', len(selected), 'model-native 128-token failures:', core['dataset_counts'])
    print('Scientific fingerprint:', stable_sha)
    print('Existing repair trial records preserved:', repairs_saved)
    return selected


SELECTED_FAILURES = choose_failures()


### 12. BD3LM model-native prefix-conditioned suffix repair, token audits, and forward-call measurement


In [ ]:
ACTIONS=('tail25','tail50','tail75','global')
REPAIR_SEEDS=(38192,38193)


def frozen_prefix_length(old_length,action):
    n=int(old_length)
    assert n>=16
    if action=='global': return 0
    fractions={'tail25':0.75,'tail50':0.50,'tail75':0.25}
    if action not in fractions:raise ValueError(action)
    return max(1,min(n-1,int(math.floor(n*fractions[action]))))

@torch.no_grad()
def self_repair(failure,action,seed):
    assert CTX and CTX['family']=='BD3LM'
    s=SRC_MAP[failure['problem_id']]
    old=[int(t) for t in failure['token_ids']]
    assert len(old)>=16
    assert not verify(failure['output'],s)['task_success']
    fixed=frozen_prefix_length(len(old),action)
    prefix=old[:fixed]
    reopen=len(old)-fixed
    prompt=prepare_ids(s)
    observed=prompt+prefix
    # Official BD3LM.sample pads the full observed prompt on the left to a
    # multiple of block_size, then appends its native diffusion-mask blocks.
    cfg=sampler_config(reopen)
    assert cfg.steps_per_block==32
    set_seed(seed)
    ts=time.perf_counter()
    torch.cuda.reset_peak_memory_stats()
    with CountCalls(CTX['model']) as calls:
        result=CTX['sampler'].sample([list(observed)],cfg,return_dict=True)
    raw=[int(t) for t in result.sequences[0].detach().cpu().tolist()]
    offset=math.ceil(len(observed)/BLOCK)*BLOCK
    assert raw[offset-len(observed):offset]==observed, 'BD3LM changed observed prefix'
    regenerated=raw[offset:]
    assert 0<len(regenerated)<=reopen
    assert CTX['mask'] not in regenerated, 'Unresolved diffusion masks'
    new_visible=visible_answer_tokens(regenerated)
    repaired=prefix+new_visible
    assert repaired[:fixed]==prefix, 'Locked response prefix changed'
    assert len(repaired)<=len(old)
    text=decode(repaired)
    v=verify(text,s)
    common=sum(int(a==b) for a,b in zip(old,repaired))
    preservation=common/max(1,max(len(old),len(repaired)))
    different=max(len(old),len(repaired))-common
    previous_num=lastnum(failure['output'])
    current_num=lastnum(text)
    numeric_recurrence=bool(not v['task_success'] and previous_num is not None and current_num is not None and equals(previous_num,current_num))
    return dict(id=f"{failure['problem_id']}|{action}|{seed}",problem_id=failure['problem_id'],
        dataset=s['dataset'],action=action,seed=int(seed),original_output=failure['output'],
        repaired_output=text,original_token_count=len(old),num_reopened=reopen,num_locked=fixed,
        preservation=float(preservation),locked_tokens_exact=True,changed_token_positions=int(different),
        wrong_answer_recurrence_numeric=numeric_recurrence,
        exact_wrong_output_recurrence=bool(not v['task_success'] and repaired==old),
        task_success=bool(v['task_success']),strict_success=bool(v['strict_success']),
        numeric_success=bool(v['numeric_success']),verification=v,
        native_method='BD3LMSampler.sample_prefix_conditioned_suffix',
        requested_steps=int(cfg.steps),requested_steps_per_block=int(cfg.steps_per_block),
        nfe=int(calls.count),wall_seconds=float(time.perf_counter()-ts),
        peak_vram_gb=float(torch.cuda.max_memory_allocated()/2**30),
        generated_suffix_tokens=int(len(new_visible)),raw_repaired_tokens=repaired)

REPAIR_JOBS=[(s,act,seed) for s in SELECTED_FAILURES for act in ACTIONS for seed in REPAIR_SEEDS]
assert len(REPAIR_JOBS)==64

def repair_progress():
    saved=completed_map(CTX['paths']['repair'],CTX['signature'])
    print('Saved BD3LM repair trials:',len(saved),'/64')

def repair_batch(start,n=4,max_minutes=8):
    assert CTX and CTX['family']=='BD3LM'
    assert 0<=start<64 and 1<=n<=4 and start+n<=64
    jobs=REPAIR_JOBS[start:start+n];path=CTX['paths']['repair'];sig=CTX['signature']
    done=completed_map(path,sig)
    began=time.monotonic()
    for failure,action,seed in jobs:
        tid=f"{failure['problem_id']}|{action}|{seed}"
        if tid in done:
            assert done[tid]['status']=='ok';print('SKIP',tid);continue
        if time.monotonic()-began>max_minutes*60:
            print('Soft time cutoff; rerun this batch to continue.');break
        try:
            row=self_repair(failure,action,seed)
            row.update(signature=sig,status='ok')
            append_record(path,row)
            print('SAVED',tid,'success=',row['task_success'],'forward calls=',row['nfe'])
        except Exception as exc:
            append_record(CTX['paths']['errors'],dict(stage='bd3lm_repair',id=tid,error=repr(exc),
                traceback=traceback.format_exc(),utc=datetime.now(timezone.utc).isoformat()))
            raise RuntimeError('BD3LM repair failed; previous trials are saved to Google Drive.') from exc
    repair_progress()


### 13. Repairs 01–32 — eight checkpointed 4-trial batches

In [ ]:
repair_batch(0,n=4,max_minutes=8)

In [ ]:
repair_batch(4,n=4,max_minutes=8)

In [ ]:
repair_batch(8,n=4,max_minutes=8)

In [ ]:
repair_batch(12,n=4,max_minutes=8)

In [ ]:
repair_batch(16,n=4,max_minutes=8)

In [ ]:
repair_batch(20,n=4,max_minutes=8)

In [ ]:
repair_batch(24,n=4,max_minutes=8)

In [ ]:
repair_batch(28,n=4,max_minutes=8)

### 14. Repairs 33–64 — eight checkpointed 4-trial batches

In [ ]:
repair_batch(32,n=4,max_minutes=8)

In [ ]:
repair_batch(36,n=4,max_minutes=8)

In [ ]:
repair_batch(40,n=4,max_minutes=8)

In [ ]:
repair_batch(44,n=4,max_minutes=8)

In [ ]:
repair_batch(48,n=4,max_minutes=8)

In [ ]:
repair_batch(52,n=4,max_minutes=8)

In [ ]:
repair_batch(56,n=4,max_minutes=8)

In [ ]:
repair_batch(60,n=4,max_minutes=8)

### 15. Problem-level statistics, source verification, and final archive


In [ ]:
import csv

def write_csv(file,rows):
    if not rows:return
    keys=list(rows[0].keys())
    with open(file,'w',newline='',encoding='utf-8') as h:
        writer=csv.DictWriter(h,fieldnames=keys,extrasaction='ignore')
        writer.writeheader();writer.writerows(rows)

def wilson95(k,n):
    z=1.95996398454;p=k/n;scale=1+z*z/n
    a=(p+z*z/(2*n))/scale
    b=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))/scale
    return max(0,a-b),min(1,a+b)

def paired_exact(old,new):
    old_only=sum(bool(a) and not bool(b) for a,b in zip(old,new))
    new_only=sum(not bool(a) and bool(b) for a,b in zip(old,new))
    n=old_only+new_only
    p=1.0 if n==0 else min(1.0,2*sum(math.comb(n,i) for i in range(min(old_only,new_only)+1))/2**n)
    return dict(old_only=old_only,new_only=new_only,discordant=n,two_sided_exact_p=p)

def finish_experiment():
    assert CTX and CTX['family']=='BD3LM'
    paths=CTX['paths'];sig=CTX['signature']
    generations=completed_map(paths['natural'],sig)
    repairs=completed_map(paths['repair'],sig)
    expected_g={f"{s['problem_id']}|fixed32x32|{b}|{GEN_SEED}" for s in SOURCES for b in BUDGETS}
    expected_r={f"{s['problem_id']}|{act}|{seed}" for s in SELECTED_FAILURES for act in ACTIONS for seed in REPAIR_SEEDS}
    assert set(generations)==expected_g and len(generations)==48, f'Incomplete generations: {len(expected_g-set(generations))} missing'
    assert set(repairs)==expected_r and len(repairs)==64, f'Incomplete repairs: {len(expected_r-set(repairs))} missing'
    assert all(x['status']=='ok' for x in list(generations.values())+list(repairs.values()))
    assert all(x['locked_tokens_exact'] for x in repairs.values())
    ids=[s['problem_id'] for s in SOURCES]
    reference_by_id={x['problem_id']:x for x in REFERENCE}
    assert set(reference_by_id)==set(ids)
    natural=list(generations.values());repair=list(repairs.values())
    generation_summary=[]
    paired=[]
    for budget in BUDGETS:
        group={x['problem_id']:x for x in natural if x['budget']==budget}
        assert set(group)==set(ids)
        new_success=sum(int(x['task_success']) for x in group.values())
        old_success=sum(int(x['task_success']) for x in REFERENCE)
        ci=wilson95(new_success,24)
        generation_summary.append(dict(model='BD3LM-0.5B',budget=int(budget),n=24,
            task_success=new_success,task_success_rate=new_success/24,
            wilson95_low=ci[0],wilson95_high=ci[1],mean_forward_calls=sum(x['nfe'] for x in group.values())/24,
            mean_wall_seconds=sum(x['wall_seconds'] for x in group.values())/24,
            baseline_64token_success=old_success,
            contrast_note='64-token baseline versus longer output AND larger total compute; descriptive only'))
        paired.append(dict(new_budget=budget,baseline_budget=64,
            **paired_exact([reference_by_id[i]['task_success'] for i in ids],
                           [group[i]['task_success'] for i in ids])))
    repair_summary=[]
    for action in ACTIONS:
        g=[r for r in repair if r['action']==action]
        assert len(g)==16
        ids8={r['problem_id'] for r in g};assert len(ids8)==8
        successes=sum(int(r['task_success']) for r in g)
        ever=sum(int(any(r['task_success'] for r in g if r['problem_id']==p)) for p in ids8)
        lo,hi=wilson95(ever,8)
        fail=[r for r in g if not r['task_success']]
        repair_summary.append(dict(action=action,trials=16,unique_failure_problems=8,
            successful_trials=successes,trial_success_rate=successes/16,
            problems_ever_repaired=ever,problem_ever_repaired_rate=ever/8,
            problem_wilson95_low=lo,problem_wilson95_high=hi,
            mean_preservation=sum(r['preservation'] for r in g)/16,
            mean_forward_calls=sum(r['nfe'] for r in g)/16,
            mean_wall_seconds=sum(r['wall_seconds'] for r in g)/16,
            same_numeric_given_failure=sum(int(r['wrong_answer_recurrence_numeric']) for r in fail)/max(1,len(fail))))
    output=paths['folder']
    write_csv(output/'controlled_length_summary.csv',generation_summary)
    write_csv(output/'paired_vs_64token_descriptive.csv',paired)
    write_csv(output/'repair_action_summary.csv',repair_summary)
    oracle=sum(int(any(r['task_success'] for r in repair if r['problem_id']==f['problem_id'])) for f in SELECTED_FAILURES)/8
    best=max(r['trial_success_rate'] for r in repair_summary)
    gate=dict(run_signature=sig,generations=len(natural),repairs=len(repair),unique_failure_problems=8,
        preservation_audits_pass=True,native_generation='official_BD3LMSampler_sample',
        native_repair='official_BD3LMSampler_suffix_reopen',
        steps_per_block=32,generation_budgets=[128,256],positive_repair_trials=sum(int(r['task_success']) for r in repair),
        best_fixed_trial_success=best,in_sample_oracle_problem_success=oracle,
        signal_ready=bool(sum(int(r['task_success']) for r in repair)>0),complete=True)
    atomic_json(output/'NB23_GATE.json',gate)
    artifacts=[paths['manifest'],paths['natural'],paths['repair'],paths['cohort'],
        output/'controlled_length_summary.csv',output/'paired_vs_64token_descriptive.csv',
        output/'repair_action_summary.csv',output/'NB23_GATE.json']
    hashes={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in artifacts}
    atomic_json(output/'artifact_manifest.json',dict(sha256=hashes,embedded_bd3lm64_reference_sha256=REFERENCE_SHA256))
    artifacts.append(output/'artifact_manifest.json')
    reconciliation = output/'generation_cohort_reconciliation_v2.json'
    if reconciliation.exists():
        artifacts.append(reconciliation)
    zip_path=Path('/content/RepairScopeX_NB23_BD3LM_ConstantPerBlock_Repair_results.zip')
    with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for p in artifacts:z.write(p,p.name)
    print('BD3LM NB23 COMPLETE:',json.dumps(gate,indent=2))
    print('Generation results',generation_summary)
    print('Repair results',repair_summary)
    return zip_path

RESULT_ZIP=finish_experiment()
from google.colab import files
files.download(str(RESULT_ZIP))


### 16. Return the results

After Section 10a you can send `RepairScopeX_NB23_BD3LM_ConstantPerBlock_Generation_only.zip`, even if repairs are unfinished. After completing all repair batches send `RepairScopeX_NB23_BD3LM_ConstantPerBlock_Repair_results.zip`.

**No model accuracy or repair outcome is assumed in advance.** If the eight-failure cohort cannot be formed, retain the generation-only archive and report the cohort-eligibility failure instead of manufacturing trials. These are exploratory results on previously frozen 24 prompts, not a fully external generalization study.
